In [1]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import time
from collections import defaultdict
import json

In [2]:
nba_sc = pd.read_csv("games.csv")
nba_sc.head(2)

,Date,Visitor,PTS,Home,PTS.1,Attend.,LOG,Arena,Notes
0,"Sat, Nov 01, 2025",Golden State Warriors,NaN,Boston Celtics,NaN,"19,000",7:30 PM,TD Garden,NaN
1,"Sat, Nov 01, 2025",Los Angeles Lakers,NaN,New York Knicks,NaN,"19,400",7:30 PM,Madison Square Garden,NaN


## 1. Write codes that computes and prints, for each team i, the following information:

In [3]:
all_teams = sorted(set(nba_sc['Home'].unique()).union(set(nba_sc['Visitor'].unique())))

for team in all_teams:
    home_games = nba_sc[nba_sc['Home'] == team].copy()
    away_games = nba_sc[nba_sc['Visitor'] == team].copy()

    print(f"{team}")
    print("" * 80)
        
#a) all the dates when team i played home
    home_dates = sorted(home_games["Date"].unique())
    print(f"\n(a) Dates for at Home Games ({len(home_dates)} games):")
    
    for date in home_dates:
        opponents = home_games[home_games["Date"] == date]["Visitor"].tolist()
        print(f"    {date}: vs {', '.join(opponents)}")
    
    #b): for each team j, the number of times team i played against team j at home
    matchups_at_home = home_games.groupby("Visitor").size().to_dict()
    print(f"\n(b) Home games vs:")
    for opponent in all_teams:
        if opponent != team:
            games = matchups_at_home.get(opponent, 0)
            if games >= 1:
                print(f"    vs {opponent:30s}")
    
    #c): for each team j, the number of times team i played against team j away (i.e., at j’s home)
    matchups_on_road = away_games.groupby("Home").size().to_dict()
    print(f"\n(c) Away games vs:")
    for opponent in all_teams:
        if opponent != team:
            games = matchups_on_road.get(opponent, 0)
            if games >= 1:
                print(f"    Visiting {opponent:30s}")
    
    #d) all the dates when team j played away.
    away_dates = sorted(away_games["Date"].unique())
    print(f"\n(d) Away games dates ({len(away_dates)} games):")
    for date in away_dates:
        opponents = away_games[away_games["Date"] == date]["Home"].tolist()
        print(f"{date}: Visiting the {', '.join(opponents)}")

Atlanta Hawks


(a) Dates for at Home Games (10 games):
    Fri, Nov 07, 2025: vs Golden State Warriors
    Fri, Nov 28, 2025: vs Phoenix Suns
    Mon, Nov 03, 2025: vs Toronto Raptors
    Mon, Nov 17, 2025: vs Milwaukee Bucks
    Sat, Nov 15, 2025: vs Boston Celtics
    Sat, Nov 29, 2025: vs Dallas Mavericks
    Sun, Nov 23, 2025: vs New York Knicks
    Thu, Dec 25, 2025: vs Chicago Bulls
    Thu, Nov 27, 2025: vs Los Angeles Lakers
    Wed, Nov 19, 2025: vs Miami Heat

(b) Home games vs:
    vs Boston Celtics                
    vs Chicago Bulls                 
    vs Dallas Mavericks              
    vs Golden State Warriors         
    vs Los Angeles Lakers            
    vs Miami Heat                    
    vs Milwaukee Bucks               
    vs New York Knicks               
    vs Phoenix Suns                  
    vs Toronto Raptors               

(c) Away games vs:
    Visiting Brooklyn Nets                 
    Visiting Chicago Bulls                 
    Visiting Clev

## 2. Write an Integer Program (without objective function) whose feasible solutions are all the feasible schedules, where a schedule is feasible if, for each team i:

(e) i plays home (possibly, against a different team) exactly on the dates computed
in (a) above;

(f) plays away (possibly, against a different team) exactly on the dates computed in
(d) above;

(g) for each team j, i plays home against team j exactly the number of times computed
in (b) above;

(h) for each team j, i plays away against team j (i.e., at j’s home) exactly the number
of times computed in (c) above.

**Sets**
- Teams: $T$ = Teams in the league (16 teams)
- Fixture Dates : $F$ = Fixture dates in the schedule (16 dates)

**Parameters**

- $\text{Home}_i$: dates when team $i$ plays at home, for each team $i \in T$
- $\text{Away}_i$: dates when team $i$ plays away, for each team $i \in T$

- $l_{ij}$: number of times team $i$ plays against team $j$ at home, $\forall$ team $i \in T$
- $v_{ij}$: number of times team $i$ plays at team $j$'s stadium, $\forall$ team $i \in T$

**Decision Variables**

$$x_{ijf} = \begin{cases} 1 & \text{if team } i \text{ hosts team } j \text{ on date } f \\ 0 & \text{otherwise} \end{cases}$$

$\forall i,j \in T, i \neq j, \forall f \in F$

(e) Games at home date constraint

For each team $i \in T$ and each fixture date $f \in F$:

$$\sum_{\substack{j \in T \\ j \neq i}} x_{ijf} = \begin{cases} 1 & \text{if } f \in \text{Home}_i \\ 0 & \text{otherwise} \end{cases}$$

Team $i$ plays exactly one home game on date $f$, if $f$ is one of their designated home dates, and plays no home games on date $f$ otherwise.

(f) Away Dates from calculation from 1.d.

For each team $i \in T$ and each fixture date $f \in F$:

$$\sum_{\substack{j \in T \\ j \neq i}} x_{jif} = \begin{cases} 1 & \text{if } f \in \text{Away}_i \\ 0 & \text{otherwise} \end{cases}$$

This is so that team $i$ plays  one away game on date $f$ if $f$ is one of their selected away dates from the data. 


(g) Home Matches counted in 1.b.

For all team fixtures $(i, j) \in T$ where $i \neq j$:

$$\sum_{f \in F} x_{ijf} = l_{ij}$$

This constraint ensures that team $i$ hosts team $j$  $l_{ij}$ times across all fixture dates, matching the original schedule.

(h) Away Matches countes in 1.c.

For all matchups $(i, j) \in T$,  $i \neq j$:

$$\sum_{f \in F} x_{jif} = v_{ij}$$

This ensures that team $i$ visits team $j$'s arena exactly $v_{ij}$ times. 